In [ ]:
import h5py
import pandas as pd

In [ ]:
f = h5py.File('GW170817_GWTC-1.hdf5', 'r')
path = 'GW170817_GWTC-1.hdf5'
print(list(f.keys()))
print(type(f['IMRPhenomPv2NRT_highSpin_posterior']))
#print(type(f['Overall_posterior']))
#print(type(f['SEOBNRv3_posterior']))
#print(type(f['prior']))
post = f['IMRPhenomPv2NRT_highSpin_posterior']
print(post.dtype)

df = pd.read_hdf('GW170817_GWTC-1.hdf5', key='IMRPhenomPv2NRT_highSpin_posterior')
print(df.columns)
print(df.head(5))

distances = df['luminosity_distance_Mpc']
cos_iota = df['costheta_jn']

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid # for cumulative integration
import math
import emcee

In [ ]:
z = 0.009877        # redshift from https://ned.ipac.caltech.edu/byname?objname=ngc+4993&hconst=67.8&omegam=0.308&omegav=0.692&wmap=4&corr_z=1
# z_uncert = 1.67e-5 do we need to take this into account?

c = 299792.458      # speed of light in km/s from https://physics.nist.gov/cgi-bin/cuu/Value?c

In [ ]:
# low redshift approximation: d_L = (c/H_0) * z
H_0 = (c * z) / distances

#Visualize the 2D distribution of H_0 and cos(iota)
plt.hist2d(H_0, cos_iota, bins=500, cmap = 'plasma', range=None, density=False, weights=None, cmin=None, cmax=None, data=None)
plt.xlabel(r'$H_0$ (km/s/Mpc)')
plt.ylabel(r'cos($\iota$)')
plt.title(r'2D Histogram of $H_0$ and cos($\iota$)')
plt.show()

#### Computation of Jacobian:
$J = \frac{\partial d_L}{\partial H_0} = -\frac{cz}{H_0^2}$,

where we have used the low redshift approximation $d_L = \frac{cz}{H_0}$

In [ ]:
# meshgrids used for evaluating the kde on a grid of points
# cos iota from -1 to 1, H_0 from min to max of the samples

H_0_grid, cos_iota_grid = np.meshgrid(np.linspace(min(H_0), max(H_0), 100), np.linspace(-1, 1, 100))

# low redshift approximation: d_L = (c/H_0) * z
d_l_grid = (c * z) / H_0_grid

In [ ]:
def likelihood (distances=distances, cos_iota=cos_iota, z=z, H_0_grid=H_0_grid, cos_iota_grid=cos_iota_grid):
    """ 
    Compute the likelihood of the data given the model parameters (H_0, cos(iota)) using a kernel density estimate (kde) of the data.
    Makes use of the low redshift approximation to relate the luminosity distance to H_0 and the redshift.
    """ 
    
    data_l_d_cos_iota = np.vstack((distances,cos_iota)) # needed because kde for 2D takes a 2D array of shape (# dims, # data)

    # kde model
    kde = gaussian_kde(data_l_d_cos_iota)

    # apply kde to the grid points
    # we need to ravel (flatten) the grid points and stack them into a 2D array of shape (# dims, # grid points) for the kde
    kde_values = kde(np.vstack([d_l_grid.ravel(), cos_iota_grid.ravel()]))

    # define jacobian for the transformation from (d_L, cos(iota)) to (H_0, cos(iota))
    J = (c * z) / (H_0_grid**2)

    # likelihood, need to reshape the kde values back to the shape of the grid and multiply by the jacobian
    # grid shape needed for 
    likelihood = kde_values.reshape(H_0_grid.shape) * J

    return likelihood

def log_likelihood (theta, distances=distances, cos_iota_data=cos_iota, z=z):
    """ 
    Compute the likelihood of the data given the model parameters (H_0, cos(iota)) using a kernel density estimate (kde) of the data.
    Makes use of the low redshift approximation to relate the luminosity distance to H_0 and the redshift.
    
    theta = [H0,cos_iota]
    """ 
    H0, cos_iota_val = theta
    
    data_l_d_cos_iota = np.vstack((distances,cos_iota)) # needed because kde for 2D takes a 2D array of shape (# dims, # data)

    # kde model
    kde = gaussian_kde(data_l_d_cos_iota)

    d_L = (c * z / H0)

    # we evaluate the KDE at this point theta
    kde_values = kde([d_L,cos_iota_val])

    # define jacobian for the transformation from (d_L, cos(iota)) to (H_0, cos(iota))
    J = (c * z) / (H0**2)

    # likelihood, need to reshape the kde values back to the shape of the grid and multiply by the jacobian
    # grid shape needed for 
    likelihood = kde_values * J

    return np.log(likelihood[0])

In [ ]:
def H_0_posterior(distances = distances, cos_iota=cos_iota, H_0_grid=H_0_grid, cos_iota_grid=cos_iota_grid):
    """ 
    Compute the posterior distribution of H_0.
    """
    
    # posterior = likelihood * prior
    # we assume a flat prior on cos(iota) and a flat-in-log prior on H_0 (proportional to 1/H_0) 
    posterior = likelihood(distances, cos_iota) * (1/H_0_grid)

    # marginalize over cos(iota) by integrating the posterior over cos(iota) from -1 to 1
    unnormalised_posterior_H0 = cumulative_trapezoid(posterior, cos_iota_grid[:,0], initial=0, axis=0)
    normalization = cumulative_trapezoid(unnormalised_posterior_H0, H_0_grid[0,:], initial=0)
    posterior_H0 = unnormalised_posterior_H0 / normalization
    
    return posterior_H0

In [ ]:
def log_prior(theta):
    H0, cos_iota = theta
    
    if 50 < H0 < 100 and -1 <= cos_iota <= 1:
        return -np.log(H0)
    return -np.inf

def log_posterior(theta, distances=distances, cos_iota_data=cos_iota, z=z):
    lp = log_prior(theta)
    
    if not np.isfinite(lp):
        return -np.inf
    
    return lp + log_likelihood(theta, distances, cos_iota_data, z)

In [ ]:
ndim = 2  
nwalkers = 32

initial = np.array([70])  # starting guess

pos = initial + 1e-2 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_posterior, args=[data]
)

sampler.run_mcmc(pos, 5000, progress=True)

samples = sampler.get_chain(discard=1000, thin=10, flat=True)

H0_samples = samples[:, 0]

median = np.median(H0_samples)
low, high = np.percentile(H0_samples, [16, 84])